# 大模型压缩：从方法选择到可部署产物

> **本章定位**：模型压缩改变模型本体、参数表示或替代模型，用于降低存储、显存、带宽、FLOPs 或延迟。

> **章节边界**：本章属于训练与推理系统：模型优化。训练优化不改变模型本体；LoRA/PEFT 减少可训练参数，但不压缩基座模型；KV Cache、Continuous Batching 和 FlashAttention 属于推理优化。

**本章总览**：围绕量化、剪枝、低秩分解与知识蒸馏建立统一方法框架，并以质量、制品大小、峰值内存、目标 Kernel、延迟和吞吐验证压缩收益。

```mermaid
flowchart LR
    M["已训练模型"] --> B["固定质量与性能基线"]
    B --> Q{"主要目标"}
    Q -->|"减少 bit 与带宽"| A["量化"]
    Q -->|"删除冗余结构"| P["剪枝"]
    Q -->|"降低矩阵秩"| L["低秩分解"]
    Q -->|"得到更小模型"| D["知识蒸馏"]
    A --> R["Calibration / Recovery"]
    P --> R
    L --> R
    D --> R
    R --> E["目标后端与 Kernel 验收"]
    E --> O["版本化压缩产物"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 训练与推理系统：模型优化；模型训练与适配的压缩专题 |
| 本章定位 | 纵向压缩模型本体或表示，并验证收益能在目标后端兑现。 |
| 先修知识 | 掌握 `31` 的模型结构、矩阵运算和精度表示；蒸馏与训练时压缩另需 `40` 的训练知识。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | 核心数值实验 CPU 可运行；真实 Kernel 验收需目标硬件。 |
| 输入 | 固定基线模型、校准数据和目标硬件约束。 |
| 交付物 | 剪枝、低秩、量化或蒸馏产物及统一质量/性能报告。 |

### 1.1．学习目标

完成本章后，读者能够区分剪枝、低秩分解、量化与知识蒸馏的优化对象，复现其核心数值路径，并验证压缩制品是否在目标后端兑现质量、内存、延迟或吞吐收益。


In [ ]:
# 准备压缩实验所需的最小 PyTorch 环境。
import copy
import random

import torch
from torch import nn
import torch.nn.functional as F

# 42 仅固定权重、输入和剪枝掩码；正式评估使用预注册的多个 Seed，并对所有方案保持一致。
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# 统一模型与张量所在设备，后续示例不绑定特定加速后端。
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print({"torch": torch.__version__, "device": str(DEVICE)})


## 2．直觉与输入输出契约

| 技术 | 是否改变基座模型 | 主要阶段 | 是否天然降低推理延迟 |
|---|---:|---|---:|
| Gradient Accumulation / Checkpointing | 否 | 训练 | 否 |
| LoRA / Adapter / Prompt Tuning | 基座不变，增加任务参数 | 参数高效训练 | 否 |
| Quantization | 改变参数或 Activation 表示 | 压缩 / 推理 | 否，依赖低精度 Kernel |
| Pruning | 删除权重或结构 | 压缩 | 只有结构或稀疏 Kernel 可利用时 |
| Low-rank Factorization | 用小矩阵近似大矩阵 | 压缩 | 只有新 GEMM 组合更高效时 |
| Distillation | 训练新的 Student | 压缩 | Student 更适合目标硬件时 |
| MoE | 通常增加总参数、稀疏激活 | 模型架构 | 不是压缩，受路由与通信限制 |
| KV Cache / FlashAttention | 模型权重不变 | 推理优化 | 在匹配负载与后端时 |

```mermaid
flowchart TD
    X["优化技术"] --> T{"改变训练过程还是模型产物？"}
    T -->|"只改变训练过程"| A["训练优化"]
    T -->|"改变模型或参数表示"| C["模型压缩"]
    T -->|"模型确定后改变执行方式"| I["推理优化"]
    A --> A1["AMP、Checkpointing、ZeRO、PEFT"]
    C --> C1["量化、剪枝、低秩、蒸馏"]
    I --> I1["KV Cache、Kernel、编译、调度、并行"]
```

压缩有效性不能仅由文件大小判断，还应同时记录质量、参数量、序列化大小、峰值内存、目标硬件延迟、吞吐和 Kernel 命中情况。


### 2.1．统一基线与评价维度

本章采用两层 MLP 观察剪枝与低秩分解。真实 Transformer 中最常见的压缩对象是 Attention Projection、FFN Linear、Embedding 和 LM Head。

- **质量基线**：固定验证集和任务指标。
- **结构基线**：参数量、层数、Hidden Size、Head 数和矩阵形状。
- **系统基线**：序列化大小、峰值内存、延迟、吞吐与功耗。
- **后端基线**：dtype、布局、Kernel、Batch 和序列长度分布。


In [ ]:
# 使用外部形状不变的两层 MLP，便于观察隐藏通道压缩。
# 64→128→32 的矩形 MLP 用于同时验证通道剪枝与低秩分解；仅控制实验成本，生产维度读取目标模型。
class MyCompressionMLP(nn.Module):
    """提供用于剪枝和低秩分解验证的两层前馈网络。"""
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=32):
        """初始化输入层与输出层，并记录中间隐藏宽度。"""
        super().__init__()
        self.up = nn.Linear(input_dim, hidden_dim)
        self.down = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """对输入依次执行线性层、GELU 和输出投影。"""
        hidden = F.gelu(self.up(x))
        return self.down(hidden)


model = MyCompressionMLP().to(DEVICE).eval()
# 固定形状：probe.shape = [8, 64]。
probe = torch.randn(8, 64, device=DEVICE)

# 记录压缩前参数量与输出，后续方法都与同一基线比较。
with torch.inference_mode():
    baseline_output = model(probe)

baseline_parameters = sum(parameter.numel() for parameter in model.parameters())
print({"baseline_parameters": baseline_parameters, "output_shape": tuple(baseline_output.shape)})


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

压缩是在明确误差预算下减少参数、位宽或计算。截断 SVD 给出固定秩下的最优 Frobenius 范数近似：

$$
W=U\Sigma V^\top,\qquad W_r=U_{:,1:r}\Sigma_{1:r,1:r}V_{:,1:r}^\top,
\qquad \|W-W_r\|_F^2=\sum_{i>r}\sigma_i^2
$$

其中，$W\in\mathbb{R}^{m\times n}$，$r\le\min(m,n)$，$\sigma_i$ 是奇异值。量化则用比例尺 $s$ 和整数范围近似权重：$q=\operatorname{clip}(\operatorname{round}(W/s),q_{\min},q_{\max})$，$\hat W=sq$。代码中的 SVD、量化器和生产 Kernel 分别对应数学近似、制品编码与实际加速；矩阵误差更小不保证端到端质量更高，必须回到任务评测和硬件基准。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．非结构化剪枝及其执行边界

Magnitude Pruning（幅值剪枝）把绝对值较小的权重置零。它能展示稀疏机制，但普通 Dense Linear 仍会执行相同形状的 GEMM。

输入包括权重与目标稀疏率，输出仍保持原有张量形状，但部分权重被置零。实际收益依赖能够利用相应稀疏模式的存储格式与 Kernel；零元素比例本身不能证明文件、内存或延迟已经改善。


In [ ]:
# 按全局权重幅值生成 Mask，并原位清零小权重。
def my_global_magnitude_prune(model, sparsity):
    """按全模型权重绝对值阈值原地置零指定比例参数，并返回剪枝统计。"""
    named_weights = [
        (name, parameter)
        for name, parameter in model.named_parameters()
        if parameter.ndim >= 2
    ]
    magnitudes = torch.cat([
        parameter.detach().abs().flatten()
        for _, parameter in named_weights
    ])
    threshold = torch.quantile(magnitudes, sparsity)

    masks = {}
    with torch.no_grad():
        for name, parameter in named_weights:
            mask = parameter.abs() > threshold
            parameter.mul_(mask)
            masks[name] = mask
    return masks


unstructured_model = copy.deepcopy(model)
# 稀疏度 0.50 删除约一半权重以产生可测误差；增大压缩潜力与质量风险，执行收益仍依赖稀疏 Kernel。
unstructured_masks = my_global_magnitude_prune(unstructured_model, sparsity=0.50)

# 零元素增加但矩阵形状与参数容器数量不变，因此不能直接宣称加速。
kept = sum(int(mask.sum()) for mask in unstructured_masks.values())
total = sum(mask.numel() for mask in unstructured_masks.values())
print({
    "measured_sparsity": 1 - kept / total,
    "parameter_count": sum(parameter.numel() for parameter in unstructured_model.parameters()),
})


### 3.2．结构化剪枝与矩阵形状

对于两层 FFN，删除中间隐藏通道时必须同时删除第一层的输出行和第二层对应的输入列，才能保持外部输入输出形状不变。

<!-- diagram:structured-pruning -->

![架构图：FFN 中间通道的结构化剪枝与 Up、Down Projection 联动形状](assets/figures/A60_model_compression/structured-pruning.svg)

[TikZ 源文件](assets/figures/A60_model_compression/structured-pruning.tex)

通道评分可以来自权重范数、Activation、Gradient、Hessian 近似或任务敏感度。简单权重范数只用于解释数据流，不应直接替代生产压缩策略。


In [ ]:
# 联合两层权重为每个隐藏通道评分，并重建更小的 Dense MLP。
def my_prune_mlp_hidden(model, keep_ratio):
    """按隐藏单元的输入输出联合重要性重建更窄的 MLP，并复制保留权重。"""
    up_score = model.up.weight.detach().norm(dim=1)
    down_score = model.down.weight.detach().norm(dim=0)
    channel_score = up_score * down_score

    keep_count = max(1, int(channel_score.numel() * keep_ratio))
    keep_index = torch.topk(channel_score, keep_count).indices.sort().values

    compressed = MyCompressionMLP(
        input_dim=model.up.in_features,
        hidden_dim=keep_count,
        output_dim=model.down.out_features,
    ).to(model.up.weight.device)

    # 第一层保留对应输出行，第二层保留同一批输入列。
    with torch.no_grad():
        compressed.up.weight.copy_(model.up.weight[keep_index])
        compressed.up.bias.copy_(model.up.bias[keep_index])
        compressed.down.weight.copy_(model.down.weight[:, keep_index])
        compressed.down.bias.copy_(model.down.bias)
    return compressed, keep_index


# 保留 50% 隐藏通道形成 Dense 结构压缩；降低比例会减少参数/FLOPs，但须按任务质量与真实延迟验收。
structured_model, kept_channels = my_prune_mlp_hidden(model, keep_ratio=0.50)
with torch.inference_mode():
    structured_output = structured_model(probe)

print({
    "kept_channels": kept_channels.numel(),
    "compressed_parameters": sum(parameter.numel() for parameter in structured_model.parameters()),
    "output_shape": tuple(structured_output.shape),
})


### 3.3．低秩分解与矩阵近似

对权重 $W$ 做截断 SVD，保留前 $r$ 个奇异值：

$$W ≈ U_r Σ_r V_r^T$$

随后把它实现为 $d_{in} → r → d_{out}$ 两个 Linear。只有当 $r(d_{in}+d_{out}) < d_{in}d_{out}$，且两个较小 GEMM 在目标硬件上更高效时，才同时获得参数和延迟收益。

Low-rank Factorization 会近似原权重；LoRA 则保留完整基座并学习低秩增量，两者不能混为一谈。


In [ ]:
# 截断 SVD，并把奇异值平方根分别吸收到两个 Linear。
def my_factorize_linear_svd(layer, rank):
    """用截断 SVD 将一个 Linear 近似为两个低秩 Linear，并保留原偏置。"""
    maximum_rank = min(layer.in_features, layer.out_features)
    if not 1 <= rank <= maximum_rank:
        raise ValueError(f"rank 必须位于 [1, {maximum_rank}]，实际为 {rank}")
    weight = layer.weight.detach().float()
    u, singular_values, vh = torch.linalg.svd(weight, full_matrices=False)

    first = nn.Linear(
        layer.in_features, rank, bias=False,
        device=layer.weight.device, dtype=layer.weight.dtype,
    )
    second = nn.Linear(
        rank, layer.out_features, bias=layer.bias is not None,
        device=layer.weight.device, dtype=layer.weight.dtype,
    )

    root_s = singular_values[:rank].sqrt()
    with torch.no_grad():
        first.weight.copy_((torch.diag(root_s) @ vh[:rank]).to(first.weight.dtype))
        second.weight.copy_((u[:, :rank] @ torch.diag(root_s)).to(second.weight.dtype))
        if layer.bias is not None:
            second.bias.copy_(layer.bias)
    return nn.Sequential(first, second)


# 固定形状：source_linear.weight.shape = [128, 64]（out_features, in_features）。
source_linear = nn.Linear(64, 128)
# Rank=16 在 128×64 权重上保留最多 16 个主方向；增大可降低重构误差，但会增加参数与 FLOPs。
factorized_linear = my_factorize_linear_svd(source_linear, rank=16)

# 同时观察参数量与权重近似误差，后续还要用任务指标和真实延迟验收。
reconstructed_weight = (factorized_linear[1].weight @ factorized_linear[0].weight).detach()
relative_error = (
    torch.linalg.norm(reconstructed_weight - source_linear.weight.detach())
    / torch.linalg.norm(source_linear.weight.detach())
)
print({
    "source_parameters": sum(parameter.numel() for parameter in source_linear.parameters()),
    "factorized_parameters": sum(parameter.numel() for parameter in factorized_linear.parameters()),
    "relative_weight_error": float(relative_error),
})


#### 3.3.1．Rank、参数比例与矩阵近似误差

学习问题是：截断 SVD 增加 Rank 时，矩阵近似误差与参数预算如何共同变化。下图对同一个 `source_linear` 逐一调用 `my_factorize_linear_svd`，不会更换权重样本。验收条件是 Rank 增加时相对权重误差不升高，因子参数比例不降低。


In [ ]:
# 使用真实权重的奇异值与逐 Rank 重构结果，呈现压缩—误差关系。
import matplotlib.pyplot as plt

candidate_ranks = [1, 2, 4, 8, 16, 32, 64]
source_parameter_count = sum(parameter.numel() for parameter in source_linear.parameters())
rank_errors = []
rank_parameter_ratios = []
for candidate_rank in candidate_ranks:
    candidate_factorization = my_factorize_linear_svd(source_linear, rank=candidate_rank)
    candidate_reconstruction = (candidate_factorization[1].weight @ candidate_factorization[0].weight).detach()
    rank_errors.append(float(
        torch.linalg.vector_norm(candidate_reconstruction - source_linear.weight.detach())
        / torch.linalg.vector_norm(source_linear.weight.detach()).clamp_min(1e-12)
    ))
    rank_parameter_ratios.append(
        sum(parameter.numel() for parameter in candidate_factorization.parameters()) / source_parameter_count
    )
if any(left < right - 1e-6 for left, right in zip(rank_errors, rank_errors[1:])):
    raise RuntimeError("SVD 重构误差未随 Rank 增加而保持非增")
if any(left > right for left, right in zip(rank_parameter_ratios, rank_parameter_ratios[1:])):
    raise RuntimeError("因子参数比例未随 Rank 增加而保持非减")
singular_values = torch.linalg.svdvals(source_linear.weight.detach().float()).cpu()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
axes[0].plot(range(1, len(singular_values) + 1), singular_values, color="#0072B2")
axes[0].axvline(16, color="#D55E00", linestyle="--", label="正文 Rank=16")
axes[0].set(title="source_linear 奇异值谱", xlabel="奇异值序号", ylabel="奇异值")
axes[0].legend()
axes[1].plot(rank_parameter_ratios, rank_errors, marker="o", color="#009E73")
for candidate_rank, ratio, error in zip(candidate_ranks, rank_parameter_ratios, rank_errors):
    axes[1].annotate(f"r={candidate_rank}", (ratio, error), xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[1].axvline(1.0, color="black", linestyle=":", label="原层参数量")
axes[1].set(title="参数比例与相对权重误差", xlabel="因子参数 / 原层参数", ylabel="Relative Frobenius Error")
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()
print({rank: {"parameter_ratio": round(ratio, 4), "relative_error": round(error, 4)} for rank, ratio, error in zip(candidate_ranks, rank_parameter_ratios, rank_errors)})


随机初始化权重通常没有快速衰减的奇异值谱，因此低 Rank 误差较高是合理结果，不应为了得到平滑曲线而更换样本。权重 Frobenius Error 只描述矩阵近似，不能替代下游质量、校准、安全和真实 Kernel 延迟；当因子参数比例超过 1 时，低秩分解已经失去参数压缩意义。


### 3.4．量化的数值表示与执行路径

> **本节定位**：量化改变权重、激活或 KV Cache 的数值表示，主线是误差控制、Calibration 与硬件 Kernel 匹配；推理侧如何消费量化资产见 [A50_inference_optimization.ipynb](A50_inference_optimization.ipynb)。

本节说明量化的数值映射、粒度、校准、误差与执行边界，并以 INT8 weight-only（仅权重量化）建立最小实现，再比较 GPTQ、AWQ、NF4 与生产后端的适用条件。

<!-- diagram:quantization-runtime-path -->

![架构图：浮点权重经校准、整数表示与受支持 Kernel 获得实际性能收益的路径](assets/figures/A60_model_compression/quantization-runtime-path.svg)

[TikZ 源文件](assets/figures/A60_model_compression/quantization-runtime-path.tex)

文件大小下降不等于推理更快；只有硬件、数据类型、布局和内核均匹配时才会获得真实加速。


#### 3.4.1．环境与基准张量


In [ ]:
# 固定随机状态并准备浮点权重，作为各量化方案的共同基准。

import random
from importlib.util import find_spec

import torch
from torch import nn
import torch.nn.functional as F

# 42 仅固定随机权重、校准输入与量化误差；正式误差结论使用预注册的多个 Seed，且校准集覆盖比 Seed 取值更重要。
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print("device:", DEVICE)


#### 3.4.2．对称 INT8 量化的原理实现

对称量化令零点为 0：

$$s=\frac{\max(|x|)}{127},\qquad q=\operatorname{clip}(\operatorname{round}(x/s),-127,127),\qquad \hat{x}=sq$$

输入是浮点张量，输出为 `int8` 张量与 scale；反量化输出近似浮点张量。常量全零张量必须显式处理，避免 scale 为 0。


In [ ]:
# 从最大绝对值计算缩放因子，把浮点张量映射到 INT8 范围。

def my_quantize_symmetric(tensor):
    """将浮点张量按单一对称尺度量化为 INT8，并返回量化值与尺度。"""
    max_value = tensor.detach().abs().amax()
    # 对称量化把最大绝对值映射到 127，零点固定为 0。
    scale = torch.clamp(max_value / 127.0, min=torch.finfo(torch.float32).eps)
    quantized = torch.clamp(torch.round(tensor / scale), -127, 127).to(torch.int8)
    return quantized, scale.to(torch.float32)

def my_dequantize_symmetric(quantized, scale):
    """使用给定尺度把 INT8 张量恢复为 FP32 近似值。"""
    return quantized.to(torch.float32) * scale

# 64×128 矩形权重同时覆盖输入/输出通道，0.2 仅使误差便于测量；真实 Scale 必须由目标权重统计得到。
weight = torch.randn(64, 128) * 0.2
qweight, scale = my_quantize_symmetric(weight)
restored = my_dequantize_symmetric(qweight, scale)
mae = (weight - restored).abs().mean()
print("scale:", float(scale), "MAE:", float(mae))


#### 3.4.3．Per-tensor 与 Per-channel 粒度

Per-tensor（每张量）共享一个 scale，元数据开销较低，但更容易受离群值影响；Per-channel（每输出通道）为每一行权重分配 scale，通常能够降低线性层误差，但会增加尺度元数据及相应 Kernel 处理开销。

<!-- diagram:quantization-granularity -->
量化粒度决定 Scale 的共享范围，也决定元数据开销与误差上限：

![架构图：权重矩阵在 Per-tensor、Per-channel 与 Per-group 粒度下共享 Scale 的范围](assets/figures/A60_model_compression/quantization-granularity.svg)

[TikZ 源文件](assets/figures/A60_model_compression/quantization-granularity.tex)


In [ ]:
# 为每个输出通道单独计算缩放因子，降低通道幅值差异造成的误差。

def my_quantize_per_output_channel(weight):
    # Linear 权重形状为 [out_features, in_features]，每行独立量化。
    """沿 Linear 输出通道分别计算对称尺度，返回 INT8 权重和 [out, 1] 尺度。"""
    max_value = weight.detach().abs().amax(dim=1, keepdim=True)
    scales = torch.clamp(
        max_value / 127.0, min=torch.finfo(torch.float32).eps
    ).to(torch.float32)
    quantized = torch.clamp(
        torch.round(weight / scales), -127, 127
    ).to(torch.int8)
    return quantized, scales

# 64 个输出通道的尺度跨 10^-2…10^1，用作 Per-channel 压力测试；该分布不代表真实模型权重。
weight = torch.randn(64, 128) * torch.logspace(-2, 1, 64).unsqueeze(1)
q_tensor, tensor_scale = my_quantize_symmetric(weight)
q_channel, channel_scales = my_quantize_per_output_channel(weight)
error_tensor = F.mse_loss(
    my_dequantize_symmetric(q_tensor, tensor_scale), weight
)
error_channel = F.mse_loss(
    my_dequantize_symmetric(q_channel, channel_scales), weight
)
print("MSE per-tensor:", float(error_tensor), "per-channel:", float(error_channel))


#### 3.4.4．通道动态范围与量化误差

学习问题是：当输出通道幅值跨越多个数量级时，共享一个 Scale 为什么会损失小幅值通道的精度。下图直接读取上一单元同一权重的逐通道动态范围与 MSE，对比 Per-tensor 和 Per-channel 反量化结果。验收条件是逐通道 MSE 的均值与已报告整体 MSE 一致，当前构造数据上 Per-channel 整体 MSE 不高于 Per-tensor。


In [ ]:
# 将同一真实权重的通道幅值与两种反量化误差逐行对齐。
import matplotlib.pyplot as plt

restored_tensor = my_dequantize_symmetric(q_tensor, tensor_scale)
restored_channel = my_dequantize_symmetric(q_channel, channel_scales)
channel_ranges = weight.detach().abs().amax(dim=1).float().cpu()
channel_mse_tensor = (restored_tensor - weight).square().mean(dim=1).float().cpu()
channel_mse_channel = (restored_channel - weight).square().mean(dim=1).float().cpu()
if abs(float(channel_mse_tensor.mean()) - float(error_tensor)) > 1e-6:
    raise RuntimeError("Per-tensor 逐通道 MSE 与整体 MSE 聚合不一致")
if abs(float(channel_mse_channel.mean()) - float(error_channel)) > 1e-6:
    raise RuntimeError("Per-channel 逐通道 MSE 与整体 MSE 聚合不一致")
if float(error_channel) > float(error_tensor):
    raise RuntimeError("当前异尺度权重上 Per-channel MSE 高于 Per-tensor")

channel_indices = range(len(channel_ranges))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
axes[0].semilogy(channel_indices, channel_ranges.clamp_min(1e-12), color="#0072B2")
axes[0].set(title="输出通道权重动态范围", xlabel="输出通道", ylabel="max |weight|（log scale）")
axes[1].semilogy(channel_indices, channel_mse_tensor.clamp_min(1e-12), color="#D55E00", label="Per-tensor")
axes[1].semilogy(channel_indices, channel_mse_channel.clamp_min(1e-12), color="#009E73", linestyle="--", label="Per-channel")
axes[1].set(title="逐通道反量化误差", xlabel="输出通道", ylabel="MSE（log scale）")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print({"per_tensor_mse": float(error_tensor), "per_channel_mse": float(error_channel), "mse_reduction": float(1.0 - error_channel / error_tensor)})


Per-channel 在该异尺度权重上降低数值误差，是因为每行独立使用整数动态范围；这不保证所有任务质量都提高，也不保证目标硬件更快。更多 Scale 会增加元数据和 Kernel 复杂度，最终格式仍需结合校准集、离群值、目标后端、实际延迟、内存带宽与端到端质量选择。


#### 3.4.5．INT8 Weight-only Linear 的原理实现

该类用于验证存储与数值误差：权重以 INT8 buffer 保存，前向时反量化后调用浮点矩阵乘法。该路径不提供低比特执行加速，因为前向过程仍未使用融合 Kernel；生产执行应迁移到 TorchAO、推理引擎或设备原生后端。

<!-- diagram:weight-only-runtime -->
Weight-only 量化只压缩权重；激活保持浮点，并由兼容内核完成解量化或融合计算：

![架构图：INT8 Weight-only 线性层中离线权重量化与浮点激活的 Kernel 执行边界](assets/figures/A60_model_compression/weight-only-runtime.svg)

[TikZ 源文件](assets/figures/A60_model_compression/weight-only-runtime.tex)


In [ ]:
# 保存 INT8 权重与缩放因子，在前向时反量化完成 Weight-only Linear。

class MyInt8WeightOnlyLinear(nn.Module):
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """保存逐输出通道 INT8 权重，并在前向时反量化执行线性变换。"""
    def __init__(self, weight, bias=None):
        """量化浮点权重并把量化值、尺度和可选偏置注册为缓冲区。"""
        super().__init__()
        qweight, scales = my_quantize_per_output_channel(weight)
        self.register_buffer("qweight", qweight)
        self.register_buffer("scales", scales)
        if bias is None:
            self.bias = None
        else:
            self.register_buffer("bias", bias.detach().clone())

    @classmethod
    def my_from_float(cls, layer):
        """从 nn.Linear 构造权重量化层；输入类型不匹配时抛出 TypeError。"""
        if not isinstance(layer, nn.Linear):
            raise TypeError("layer must be nn.Linear")
        return cls(layer.weight.detach(), layer.bias.detach() if layer.bias is not None else None)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x):
        """按输入数据类型反量化权重，并返回线性投影结果。"""
        weight = self.qweight.to(x.dtype) * self.scales.to(x.dtype)
        return F.linear(x, weight, self.bias)

# 固定形状：float_layer.weight.shape = [64, 128]（out_features, in_features）。
float_layer = nn.Linear(128, 64).eval()
quant_layer = MyInt8WeightOnlyLinear.my_from_float(float_layer).eval()
# 固定形状：x.shape = [16, 128]。
x = torch.randn(16, 128)
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    float_output = float_layer(x)
    quant_output = quant_layer(x)
relative_error = (
    (float_output - quant_output).pow(2).mean().sqrt()
    / float_output.pow(2).mean().sqrt()
)
float_bytes = float_layer.weight.numel() * float_layer.weight.element_size()
quant_bytes = (
    quant_layer.qweight.numel() * quant_layer.qweight.element_size()
    + quant_layer.scales.numel() * quant_layer.scales.element_size()
)
print(f"weight bytes: {float_bytes} → {quant_bytes}; relative RMSE={relative_error:.4f}")


#### 3.4.6．激活校准与离群值

静态激活量化需要代表性校准集。极端离群值会扩大 scale、浪费大部分整数区间。本节 observer 用于说明 min/max 校准；生产工具通常提供直方图、百分位或误差最小化 observer。


In [ ]:
# 观测激活最小值和最大值，为非对称激活量化计算 scale 与 zero point。

class MyMinMaxObserver:
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    """累计校准激活的全局最小值与最大值，用于非对称 INT8 量化。"""
    def __init__(self):
        """初始化尚未观测数据的最小值和最大值状态。"""
        self.min_value = None
        self.max_value = None

    def my_update(self, tensor):
        """从输入张量更新累计激活范围，不保留计算图。"""
        min_value = float(tensor.detach().amin())
        max_value = float(tensor.detach().amax())
        self.min_value = min_value if self.min_value is None else min(self.min_value, min_value)
        self.max_value = max_value if self.max_value is None else max(self.max_value, max_value)

    def my_scale_zero_point(self):
        """根据累计范围计算 INT8 scale 与 zero point；未校准时抛出 RuntimeError。"""
        if self.min_value is None:
            raise RuntimeError("observer has not seen calibration data")
        # 1e-12 只防止常量激活使 Scale 为零；若该下限频繁生效，应诊断校准覆盖或异常分布。
        scale = max((self.max_value - self.min_value) / 255.0, 1e-12)
        zero_point = round(-self.min_value / scale) - 128
        return scale, max(-128, min(127, zero_point))

observer = MyMinMaxObserver()
# 10 个 32×128 Batch 在控制成本的同时比单批更能覆盖范围；生产校准批数由目标分布上的统计收敛性决定。
for _ in range(10):
    observer.my_update(torch.randn(32, 128))
activation_scale, zero_point = observer.my_scale_zero_point()
print("activation scale/zero-point:", activation_scale, zero_point)


#### 3.4.7．常见大模型量化方法

| 方法 | 典型场景 | 是否需要校准数据 | 核心特点 |
|---|---|---:|---|
| INT8 weight-only | 通用推理 | 否/少量 | 风险低、后端支持广 |
| GPTQ | GPU 低比特推理 | 是 | 逐层重构误差，常见 INT4 |
| AWQ | GPU 低比特推理 | 是 | 保护显著权重通道 |
| NF4 | QLoRA 训练 | 通常否 | 针对近似正态分布权重的 4-bit 编码 |
| QAT | 高精度边缘部署 | 需要训练 | 训练时模拟量化误差 |

量化方案依次由目标硬件与推理引擎、可用 Kernel、精度预算和校准数据约束，算法名称位于这些条件之后。


#### 3.4.8．与 TorchAO 对齐

TorchAO 是 PyTorch 量化与稀疏化的主流库。API 会随版本演进，以下单元按 `../requirements.txt` 的版本探测能力；能力不受支持时显式报告状态，不以浮点回退结果表示低比特执行收益。


In [ ]:
# 检测 TorchAO 能力后切换到标准 INT8 Weight-only 量化接口。

if find_spec("torchao") is None:
    print("未安装 torchao；请按 ../requirements.txt 安装后重新运行本单元。")
else:
    # 将可选依赖或平台能力隔离处理，不影响其余验证路径。
    try:
        from torchao.quantization import Int8WeightOnlyConfig, quantize_

        # 128→128→64 MLP 与 4×128 Probe 仅验证 TorchAO API；能否加速取决于目标后端的 INT8 Kernel。
        # 固定形状：model[0].weight.shape = [128, 128]（out_features, in_features）；model[2].weight.shape = [64, 128]（out_features, in_features）。
        model = nn.Sequential(nn.Linear(128, 128), nn.GELU(), nn.Linear(128, 64)).eval()
        quantize_(model, Int8WeightOnlyConfig())
        # 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
        with torch.no_grad():
            output = model(torch.randn(4, 128))
        print("TorchAO INT8 weight-only output shape：", tuple(output.shape))
    except (ImportError, AttributeError, NotImplementedError) as error:
        print("当前 TorchAO/设备组合不支持此配置：", type(error).__name__, error)


#### 3.4.9．性能基准与误差验证

- 在目标硬件上分别测 TTFT（Time To First Token，首 token 延迟）、TPOT（Time Per Output Token，单输出 token 时间）、吞吐和峰值内存；
- 预热后同步设备再计时，固定 batch、输入长度、输出长度和采样配置；
- 对照 FP16/BF16 基线，评估 perplexity、任务集、安全集和长上下文；
- 记录模型 ID、权重哈希、量化配置、校准集版本、后端、内核和硬件型号；
- 权重减少约 4 倍并不保证端到端加速 4 倍，tokenizer、调度和 KV Cache 可能成为瓶颈。


### 3.5．知识蒸馏的原理实现

> **本节定位**：知识蒸馏通过训练新的 Student 完成模型压缩，重点是知识传递目标、结构选择与输出契约；训练系统优化见 [A40_training_optimization.ipynb](A40_training_optimization.ipynb)。

知识蒸馏（Knowledge Distillation）让小型 student（学生模型）学习大型 teacher（教师模型）的软概率、隐藏状态或生成序列。

$$L=(1-\alpha)L_{hard}+\alpha T^2\operatorname{KL}\left(p_T^{teacher}\parallel p_T^{student}\right)$$

本节围绕温度、KL 散度、`T²` 缩放、教师冻结、特征维度对齐及语言模型蒸馏的数据契约展开。

<!-- diagram:distillation-overview -->
知识蒸馏让冻结教师和可训练学生读取同一批输入，再组合软目标与真实标签监督：

![架构图：冻结 Teacher 与可训练 Student 共享输入并组合软目标和真实标签](assets/figures/A60_model_compression/distillation-overview.svg)

[TikZ 源文件](assets/figures/A60_model_compression/distillation-overview.tex)


#### 3.5.1．环境与可复现数据


In [ ]:
# 固定随机状态并生成可重复的分类数据。

import random

import torch
from torch import nn
import torch.nn.functional as F

# 42 仅固定合成数据与初始化；Hard-label/KD 对照使用相同初始权重，并在预注册的多个 Seed 上报告分布。
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
# 选择当前可用设备，后续张量和模型统一放到同一计算后端。
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

# 1200×20 输入与 5 类输出足以形成非平凡分类；0.15 是加性噪声尺度，不代表真实蒸馏数据分布。
# 固定形状：x.shape = [1200, 20]。
x = torch.randn(1200, 20)
# 固定形状：rule.shape = [20, 5]。
rule = torch.randn(20, 5)
y = (x @ rule + 0.15 * torch.randn(1200, 5)).argmax(dim=-1)
# 1000/200（约 83%/17%）为小数据固定切分；真实数据须分层或分组，并记录数据版本。
train_x, valid_x = x[:1000].to(DEVICE), x[1000:].to(DEVICE)
train_y, valid_y = y[:1000].to(DEVICE), y[1000:].to(DEVICE)
print("device:", DEVICE, "train/valid:", len(train_x), len(valid_x))


#### 3.5.2．教师模型训练与冻结

教师必须先达到可靠质量。蒸馏阶段使用 `eval()` 与 `torch.no_grad()`，避免 Dropout 引入不稳定，也避免无意义地为教师保存梯度。


In [ ]:
# 先训练容量更大的教师网络，再冻结参数提供软目标。
from tqdm.auto import trange

# 20→128→64→5 的教师明显大于学生；结构只用于建立稳定合成决策边界，真实教师须先独立验收并冻结。
class MyTeacher(nn.Module):
    """定义容量较大的分类教师网络，并可返回中间隐藏表示。"""
    def __init__(self):
        """初始化两层特征提取网络和五分类输出头。"""
        super().__init__()
        # 固定形状：self.features[0].weight.shape = [128, 20]（out_features, in_features）；self.features[2].weight.shape = [64, 128]（out_features, in_features）。
        self.features = nn.Sequential(
            nn.Linear(20, 128), nn.GELU(), nn.Linear(128, 64), nn.GELU()
        )
        # 固定形状：self.head.weight.shape = [5, 64]（out_features, in_features）。
        self.head = nn.Linear(64, 5)

    def forward(self, x, return_hidden=False):
        """计算分类 logits，并按需同时返回最后一层隐藏表示。"""
        hidden = self.features(x)
        logits = self.head(hidden)
        return (logits, hidden) if return_hidden else logits

teacher = MyTeacher().to(DEVICE)
# lr=3e-3 与 120 步让小教师学到可测边界；真实配方按验证曲线决定预算。
# betas、eps 与 weight decay 显式固定优化器契约，模型、Batch 或数据变化时须与学习率联合重调。
optimizer = torch.optim.AdamW(
    teacher.parameters(), lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)
teacher_progress = trange(120, desc="训练蒸馏教师", unit="step", dynamic_ncols=True)
for _ in teacher_progress:
    logits = teacher(train_x)
    loss = F.cross_entropy(logits, train_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    teacher_progress.set_postfix(loss=f"{float(loss.detach()):.4f}")

teacher.eval()
# 遍历可训练参数，统一处理梯度、更新或状态转换。
for parameter in teacher.parameters():
    parameter.requires_grad = False
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    teacher_accuracy = (teacher(valid_x).argmax(-1) == valid_y).float().mean()
print("teacher accuracy:", float(teacher_accuracy))


#### 3.5.3．Logit Distillation Loss 的原理实现

温度 `T>1` 展开类别概率，以呈现类别之间的相对关系。PyTorch `kl_div` 要求输入为学生模型的 log-probability，target 为教师模型的 probability；交换两者会改变 KL 散度的优化方向。`T²` 用于补偿温度造成的梯度缩小。

<!-- diagram:distillation-loss -->
温度 T 软化类别分布，学生同时学习类别关系和真实标签：

![架构图：温度软化后的 Teacher Probability、Student Log-probability 与硬标签联合损失](assets/figures/A60_model_compression/distillation-loss.svg)

[TikZ 源文件](assets/figures/A60_model_compression/distillation-loss.tex)


In [ ]:
# 组合真实标签损失与温度缩放后的 KL 散度，得到蒸馏目标。

# temperature=3 软化类别分布，alpha=0.7 将 70% 权重分给软目标、30% 保留真标签；二者须联合调参。
# 更高温度会使分布更平缓，alpha 过大可能削弱真实标签约束。
def my_distillation_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=3.0,
    alpha=0.7,
):
    """组合真实标签交叉熵与温度缩放 KL 散度，返回总损失及分项诊断。"""
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    # 硬标签损失保持任务目标，软标签损失传递教师的类别相似性。
    hard_loss = F.cross_entropy(student_logits, labels)
    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=-1
    )
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    soft_loss = F.kl_div(
        student_log_probs, teacher_probs, reduction="batchmean"
    ) * (temperature ** 2)
    # 乘回 temperature²，补偿温度缩放对梯度幅度的影响。
    total = (1 - alpha) * hard_loss + alpha * soft_loss
    return total, {"hard": hard_loss.detach(), "soft": soft_loss.detach()}

# 固定形状：test_student.shape = [8, 5]。
test_student = torch.randn(8, 5, requires_grad=True)
# 固定形状：test_teacher.shape = [8, 5]。
test_teacher = torch.randn(8, 5)
test_labels = torch.randint(0, 5, (8,))
test_loss, parts = my_distillation_loss(
    test_student, test_teacher, test_labels
)
# 反向传播计算梯度，供随后的参数更新使用。
test_loss.backward()
print(parts)


#### 3.5.4．学生模型训练

师生模型接收相同输入，并共享输出类别空间；学生模型可以采用更小的隐藏维度。实验同时训练仅使用 hard label 的基线，以隔离教师软目标对质量增量的贡献。


In [ ]:
# 训练更小的学生模型，同时学习硬标签和教师概率分布。

# 20→24→5 学生显著小于教师；真实隐藏宽度由质量、延迟和目标硬件共同决定。
class MyStudent(nn.Module):
    """定义用于对照知识蒸馏效果的紧凑分类学生网络。"""
    def __init__(self):
        """初始化单层紧凑特征网络和五分类输出头。"""
        super().__init__()
        # 固定形状：self.features[0].weight.shape = [24, 20]（out_features, in_features）。
        self.features = nn.Sequential(nn.Linear(20, 24), nn.GELU())
        # 固定形状：self.head.weight.shape = [5, 24]（out_features, in_features）。
        self.head = nn.Linear(24, 5)

    def forward(self, x, return_hidden=False):
        """计算学生分类 logits，并按需同时返回隐藏表示。"""
        hidden = self.features(x)
        logits = self.head(hidden)
        return (logits, hidden) if return_hidden else logits

def my_train_student(use_distillation, initial_state):
    """从固定初始状态训练学生，可选择蒸馏或纯硬标签目标，并返回模型与损失轨迹。"""
    student = MyStudent().to(DEVICE)
    student.load_state_dict(initial_state)
    # lr=5e-3、100 步用于短程比较 Hard/Soft Loss；优化器与训练预算须按学生验证曲线重调。
    optimizer = torch.optim.AdamW(
        student.parameters(), lr=5e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
    )
    losses = []
    route_name = "训练蒸馏学生" if use_distillation else "训练硬标签基线"
    student_progress = trange(100, desc=route_name, unit="step", dynamic_ncols=True)
    for _ in student_progress:
        student_logits = student(train_x)
        if use_distillation:
            # 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
            with torch.no_grad():
                teacher_logits = teacher(train_x)
            loss, _ = my_distillation_loss(
                student_logits, teacher_logits, train_y,
                temperature=3.0, alpha=0.7,
            )
        else:
            loss = F.cross_entropy(student_logits, train_y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
        student_progress.set_postfix(loss=f"{losses[-1]:.4f}")
    return student.eval(), losses

# 两条训练路线从完全相同的学生初始权重开始，避免把初始化差异误算成蒸馏收益。
initial_student = MyStudent().to(DEVICE)
initial_student_state = {name: value.detach().clone() for name, value in initial_student.state_dict().items()}
student, kd_losses = my_train_student(True, initial_student_state)
baseline, baseline_losses = my_train_student(False, initial_student_state)
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    student_accuracy = (student(valid_x).argmax(-1) == valid_y).float().mean()
    baseline_accuracy = (baseline(valid_x).argmax(-1) == valid_y).float().mean()
print("KD/baseline accuracy:", float(student_accuracy), float(baseline_accuracy))


#### 3.5.5．隐藏状态蒸馏

教师和学生隐藏维度不同，需用可训练投影对齐。生产中还可对齐 Attention map 或指定层；层映射必须显式记录，不能依赖“同名层”。

<!-- diagram:hidden-state-distillation -->
当师生隐藏维度或层数不同时，需要先建立层映射和投影，再计算表示损失：

![架构图：教师层映射、学生表示投影与 Token Mask 对齐后的隐藏状态蒸馏](assets/figures/A60_model_compression/hidden-state-distillation.svg)

[TikZ 源文件](assets/figures/A60_model_compression/hidden-state-distillation.tex)


In [ ]:
# 增加投影层对齐教师与学生隐藏维度，再计算特征蒸馏损失。

# 24→64 投影对齐师生隐藏维；该层映射必须随结构版本记录。
# 固定形状：projection.weight.shape = [64, 24]（out_features, in_features）。
projection = nn.Linear(24, 64).to(DEVICE)
# lr=1e-3 与 30 步用于有限的表示对齐；训练预算须与任务质量共同验收。
feature_optimizer = torch.optim.AdamW(
    list(student.parameters()) + list(projection.parameters()),
    lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)
student.train()
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    _, teacher_hidden = teacher(train_x, return_hidden=True)
feature_progress = trange(30, desc="对齐师生表示", unit="step", dynamic_ncols=True)
for _ in feature_progress:
    student_logits, student_hidden = student(
        train_x, return_hidden=True
    )
    feature_loss = F.mse_loss(
        projection(student_hidden), teacher_hidden
    )
    task_loss = F.cross_entropy(student_logits, train_y)
    # 特征损失权重 0.1 使其作为辅助项；权重过大会压过主任务，须按验证质量联合选择。
    loss = task_loss + 0.1 * feature_loss
    feature_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    feature_optimizer.step()
    feature_progress.set_postfix(
        feature_loss=f"{float(feature_loss.detach()):.4f}",
        task_loss=f"{float(task_loss.detach()):.4f}",
    )
print("final feature/task loss:", float(feature_loss), float(task_loss))


#### 3.5.6．Transformers 输出契约对齐

随机初始化的小型 BERT 用于验证 teacher/student 的 `logits` 接口，实验不依赖外部模型权重。真实训练应缓存教师 logits 或在线计算；缓存可节省算力，但必须绑定样本顺序、teacher 模型 ID、权重哈希、温度和 tokenizer 版本。


In [ ]:
# 使用 Transformers 分类模型输出 logits，衔接标准蒸馏训练接口。

from transformers import BertConfig, BertForSequenceClassification

# vocab=128、4 头及教师 64 维/4 层、学生 32 维/2 层构成离线接口夹具；宽度须整除头数，不能作为生产 BERT 配置。
teacher_config = BertConfig(
    vocab_size=128, hidden_size=64, num_hidden_layers=4,
    num_attention_heads=4, intermediate_size=128, num_labels=5,
)
student_config = BertConfig(
    vocab_size=128, hidden_size=32, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=64, num_labels=5,
)
teacher_model = BertForSequenceClassification(teacher_config).eval()
student_model = BertForSequenceClassification(student_config)
# 固定形状：input_ids.shape = [3, 12]。
input_ids = torch.randint(0, 128, (3, 12))
attention_mask = torch.ones_like(input_ids)
labels = torch.randint(0, 5, (3,))
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    teacher_logits = teacher_model(
        input_ids=input_ids, attention_mask=attention_mask
    ).logits
student_logits = student_model(
    input_ids=input_ids, attention_mask=attention_mask
).logits
loss, parts = my_distillation_loss(student_logits, teacher_logits, labels)
print("Transformers KD loss parts:", parts)


#### 3.5.7．大语言模型蒸馏路线

1. **Token-level**：对齐每个位置的完整词表 logits，信息最丰富但存储和计算昂贵；teacher/student 必须共享或映射词表。
2. **Sequence-level**：教师生成答案，学生做监督微调；工程简单，但只保留采样后的有限信息。
3. **Feature/Attention-level**：对齐隐藏状态或注意力，需要层映射与投影，适合结构相近模型。

如果使用闭源教师 API，只能在许可、隐私和服务条款允许的范围内保存生成数据；不得把敏感生产数据发送到未经批准的外部服务。

#### 3.5.8．蒸馏证据清单

- 同时保留同尺寸 hard-label 基线，证明蒸馏带来净收益；
- 记录 teacher/student 模型 ID、权重哈希、tokenizer、模板、温度、alpha 和层映射；
- 评估质量、校准误差、鲁棒性、安全性、延迟、吞吐和内存，不以参数量作为唯一判断依据；
- 教师 logits 缓存应包含样本 ID 和校验和，防止错位；
- 蒸馏可能继承并放大教师偏差，必须使用独立人工标注测试集审计。


## 4．证据验证

### 4.1．组合压缩的增量验证

```mermaid
flowchart LR
    B["原始模型与固定基线"] --> S{"需要改变架构吗？"}
    S -->|"需要更小 Student"| D["先蒸馏"]
    S -->|"保持主体架构"| P["结构化剪枝 / 低秩分解"]
    D --> R["Recovery Training"]
    P --> R
    R --> Q["按目标 Kernel 量化"]
    Q --> E["质量 + 内存 + 延迟验收"]
    E --> V["版本化模型、Tokenizer<br/>量化与结构元数据"]
```

组合方案可按以下原则逐项验证：

1. 若目标是显著缩小架构，先蒸馏到合适 Student，再做其他压缩。
2. 结构化剪枝和低秩分解改变矩阵形状，通常需要 Recovery Training。
3. 量化往往放在结构稳定后进行，避免重复 Calibration。
4. 非结构化稀疏只有目标后端支持相同稀疏模式时才进入生产路径。
5. 每增加一种压缩方法，均需单独保留质量与性能增量，以便定位误差来源。


## 5．迁移到生产库

原理实现用于固定数学语义，生产路径则由目标后端支持的制品格式与 Kernel 决定。迁移时应保持相同的基线数据、任务指标、输入形状与性能测量条件。

| 原理对象 | 生产组件 | 迁移验证 |
|---|---|---|
| 非结构化或结构化剪枝 | PyTorch 剪枝接口、目标后端稀疏工具 | 稀疏模式、序列化格式、Kernel 命中与任务质量 |
| 低秩分解 | PyTorch 线性代数接口与重构后的模型模块 | 权重重构误差、参数量、实际 GEMM 延迟与恢复训练结果 |
| 权重和激活量化 | TorchAO 与目标推理后端 | dtype、粒度、校准数据、布局、Kernel 及端到端质量 |
| Logit/Hidden-state 蒸馏 | PyTorch、Transformers 与训练编排组件 | Teacher/Student 模型 ID、权重哈希、Tokenizer、层映射、损失尺度与评测协议 |

库迁移不改变压缩目标。若后端在执行前恢复为完整稠密或高精度表示，文件体积下降不能作为运行时收益证据。


## 6．生产边界

### 6.1．生产验收清单

- 固定原始模型、数据集、Tokenizer、任务指标和推理负载。
- 分别记录参数量、文件大小、权重显存、KV Cache、Workspace 和峰值内存。
- 在目标硬件、目标 Batch 和长度分布上测 TTFT、TPOT 与吞吐。
- 检查压缩格式是否直接进入 Kernel，避免先完整解压再计算。
- 结构变化后重新检查权重共享、Embedding/LM Head 绑定、配置与序列化。
- 量化记录 dtype、粒度、Group Size、Calibration 数据与后端版本。
- 剪枝记录 Mask 或新结构；蒸馏记录 Teacher、Student 和训练数据版本。
- 保存未压缩基线、每个中间产物和回滚路径。

### 6.2．参考资料

- [PyTorch Pruning Tutorial](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html)
- [PyTorch Quantization with TorchAO](https://docs.pytorch.org/ao/stable/)
- [PyTorch Knowledge Distillation Tutorial](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
- [Hugging Face Transformers Optimization Overview](https://huggingface.co/docs/transformers/main/optimization_overview)

生产使用前，以目标硬件与锁定依赖版本对应的官方文档为准。


### 6.3．方法总结

模型压缩围绕四条主线：

- **量化**改变数值表示；
- **剪枝**删除权重或结构；
- **低秩分解**用更小矩阵近似原矩阵；
- **蒸馏**训练新的 Student。

判断标准不止于参数量降低，而是质量损失可接受、产物可恢复、目标 Kernel 可直接执行，并在真实负载下兑现内存、延迟、吞吐或成本收益。
